# KKBOX Churn Prediction — Model Training

This notebook trains and evaluates machine learning models to predict user churn.

We follow this pipeline:
1. Rebuild the feature matrix (from Preprocessing)
2. Train/Validation split
3. Baseline: Logistic Regression
4. XGBoost
5. LightGBM
6. Cross-Validation
7. Feature Importance
8. Confusion Matrix
9. ROC Curve
10. Summary

## Step 0 — Configuration & Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score, confusion_matrix, classification_report, RocCurveDisplay
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings("ignore")

# --- Global constants (mirrors src/config.py) ---
TARGET       = "is_churn"   # column we want to predict
ID_COL       = "msno"       # user identifier, not a feature
TEST_SIZE    = 0.2          # 20% of data held out for validation
RANDOM_STATE = 42           # seed for reproducibility

pd.set_option("display.max_columns", None)
print("All libraries loaded successfully.")

All libraries loaded successfully.


## Step 1 — Rebuild Feature Matrix

We repeat the full preprocessing pipeline here so the notebook is self-contained.
See  for detailed explanations of each step.

In [2]:
# --- Load raw data ---
train        = pd.read_csv("../data/train_v2.csv")
members      = pd.read_csv("../data/members_v3.csv")
transactions = pd.read_csv("../data/transactions_v2.csv")

print(f"Train: {train.shape}, Members: {members.shape}, Transactions: {transactions.shape}")

Train: (970960, 2), Members: (6769473, 6), Transactions: (1431009, 9)


In [ ]:
# --- Clean members ---
members["bd"]     = members["bd"].clip(10, 80)
members["gender"] = members["gender"].fillna("unknown")

# --- Clean transactions ---
transactions = transactions.drop_duplicates()
transactions["actual_amount_paid"] = transactions["actual_amount_paid"].clip(0, 5000)
transactions["transaction_date"] = pd.to_datetime(
    transactions["transaction_date"].astype(str), format="%Y%m%d", errors="coerce")
transactions["membership_expire_date"] = pd.to_datetime(
    transactions["membership_expire_date"].astype(str), format="%Y%m%d", errors="coerce")

# --- Discount & expiry-transaction interval ---
transactions["discount"]            = transactions["plan_list_price"] - transactions["actual_amount_paid"]
transactions["discount_rate"]        = transactions["discount"] / (transactions["plan_list_price"] + 1)
transactions["expiry_txn_interval"]  = (transactions["membership_expire_date"] - transactions["transaction_date"]).dt.days

# --- Basic transaction aggregations ---
trans_agg = transactions.groupby("msno").agg({
    "actual_amount_paid": ["mean", "sum"],
    "payment_plan_days":  ["mean"],
    "is_cancel":          ["sum"],
    "is_auto_renew":      ["mean"]
})
trans_agg.columns = ["_".join(c) for c in trans_agg.columns]
trans_agg = trans_agg.reset_index()
trans_agg["cancel_rate"]     = trans_agg["is_cancel_sum"]          / (trans_agg["payment_plan_days_mean"] + 1)
trans_agg["payment_per_day"] = trans_agg["actual_amount_paid_sum"] / (trans_agg["payment_plan_days_mean"] + 1)

# --- Advanced transaction aggregations ---
adv_agg = transactions.groupby("msno").agg({
    "discount":            ["mean", "sum"],
    "discount_rate":       ["mean"],
    "expiry_txn_interval": ["mean", "max"],
    "transaction_date":    ["count"],
})
adv_agg.columns = ["_".join(c) for c in adv_agg.columns]
adv_agg = adv_agg.reset_index().rename(columns={"transaction_date_count": "total_transactions"})

# --- Last transaction features ---
last_txn = (
    transactions.sort_values("transaction_date")
    .groupby("msno").tail(1)
    [["msno","is_cancel","is_auto_renew","payment_plan_days",
      "actual_amount_paid","discount","expiry_txn_interval"]]
    .rename(columns={
        "is_cancel":           "last_is_cancel",
        "is_auto_renew":       "last_is_auto_renew",
        "payment_plan_days":   "last_plan_days",
        "actual_amount_paid":  "last_amount_paid",
        "discount":            "last_discount",
        "expiry_txn_interval": "last_expiry_interval",
    })
)

# --- Temporal features ---
REFERENCE_DATE = pd.to_datetime("2017-03-01")
last_trans = (
    transactions.sort_values("membership_expire_date")
    .groupby("msno").tail(1)
    [["msno", "membership_expire_date", "transaction_date"]]
)
last_trans["days_left"]             = (last_trans["membership_expire_date"] - REFERENCE_DATE).dt.days
last_trans["days_since_last_txn"]   = (REFERENCE_DATE - last_trans["transaction_date"]).dt.days
temporal_feat = last_trans[["msno", "days_left", "days_since_last_txn"]]

# --- User log features (chunk processing) ---
import os
LOG_FILE = "../data/user_logs_v2.csv"
if os.path.exists(LOG_FILE):
    chunk_list = []
    for chunk in pd.read_csv(LOG_FILE, chunksize=5_000_000):
        agg = chunk.groupby("msno").agg({
            "total_secs": ["sum","mean"], "num_unq": ["sum"],
            "num_25":["sum"],"num_50":["sum"],"num_75":["sum"],
            "num_985":["sum"],"num_100":["sum"],"date":["count"]
        })
        agg.columns = ["_".join(c) for c in agg.columns]
        chunk_list.append(agg)
    logs_feat = pd.concat(chunk_list).groupby(level=0).sum().reset_index()
    logs_feat = logs_feat.rename(columns={"msno":"msno","date_count":"active_days"})
    total_songs = (logs_feat["num_25_sum"]+logs_feat["num_50_sum"]+
                   logs_feat["num_75_sum"]+logs_feat["num_985_sum"]+logs_feat["num_100_sum"])
    logs_feat["completion_rate"]    = logs_feat["num_100_sum"] / (total_songs + 1)
    logs_feat["avg_secs_per_day"]   = logs_feat["total_secs_sum"] / (logs_feat["active_days"] + 1)
    print(f"Log features: {logs_feat.shape}")
else:
    logs_feat = None
    print("user_logs_v2.csv not found — skipping log features")

# --- Merge all ---
df = train.merge(members,      on="msno", how="left")
df = df.merge(trans_agg,        on="msno", how="left")
df = df.merge(adv_agg,          on="msno", how="left", suffixes=("","_adv"))
df = df.merge(last_txn,         on="msno", how="left")
df = df.merge(temporal_feat,    on="msno", how="left")
if logs_feat is not None:
    df = df.merge(logs_feat, on="msno", how="left")

# --- Encode & fill ---
df = pd.get_dummies(df, columns=["city", "gender", "registered_via"], dummy_na=True)
df = df.fillna(0)

print(f"Feature matrix: {df.shape[0]:,} rows x {df.shape[1]} cols")
print(f"Features: {df.shape[1]-2} (excl. msno, is_churn)")

## Step 2 — Train / Validation Split

We split the data into:
- **Training set (80%)** — used to fit the model
- **Validation set (20%)** — used to evaluate performance on unseen data

 ensures both splits have the same churn ratio (~9%), which is important for imbalanced datasets.

We drop  (user ID) because it is just an identifier, not a predictive feature.

In [ ]:
# Separate features (X) and target (y)
X = df.drop(columns=[TARGET, ID_COL])  # all columns except target and user ID
y = df[TARGET]

# Stratified split: preserves the 9%/91% churn ratio in both sets
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y          # <-- critical for imbalanced data
)

print(f"Training set   : {X_train.shape[0]:,} rows")
print(f"Validation set : {X_val.shape[0]:,} rows")
print(f"Features       : {X_train.shape[1]}")
print("")
print(f"Churn rate in train : {y_train.mean():.4f}")
print(f"Churn rate in val   : {y_val.mean():.4f}")

## Step 3 — Baseline: Logistic Regression

Before training complex models, we always establish a **baseline** with a simple model.
This gives us a reference point — if XGBoost does not beat logistic regression, something is wrong.

**Log Loss** is the official competition metric:
- Lower is better
- Measures how confident and correct the predicted probabilities are
- A random model gives log_loss ≈ 0.30 for 9% churn rate

**ROC-AUC** measures the model's ability to rank churners above non-churners:
- 0.5 = random, 1.0 = perfect

In [ ]:
# Logistic Regression requires feature scaling (XGBoost/LightGBM do not)
scaler  = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit on train, transform both
X_val_scaled   = scaler.transform(X_val)

lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train_scaled, y_train)

lr_proba    = lr.predict_proba(X_val_scaled)[:, 1]  # probability of churn
lr_logloss  = log_loss(y_val, lr_proba)
lr_auc      = roc_auc_score(y_val, lr_proba)

print(f"Logistic Regression")
print(f"  Log Loss : {lr_logloss:.4f}")
print(f"  ROC-AUC  : {lr_auc:.4f}")

## Step 4 — XGBoost

XGBoost (Extreme Gradient Boosting) builds an ensemble of decision trees sequentially,
where each tree corrects the errors of the previous one.

**Key hyperparameters:**
| Parameter | Value | Meaning |
|---|---|---|
|  | 200 | Number of trees |
|  | 6 | Max depth per tree (controls overfitting) |
|  | 0.05 | Step size — smaller = more conservative |
|  | 0.8 | 80% of rows sampled per tree (reduces overfitting) |
|  | 0.8 | 80% of features sampled per tree |
|  | neg/pos | Compensates for class imbalance |

 tells XGBoost to penalize missing churners more heavily.
It is set to the ratio of non-churners to churners.

In [ ]:
# Compute class weight to handle imbalance
# scale_pos_weight = count(negative class) / count(positive class)
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count
print(f"scale_pos_weight = {neg_count} / {pos_count} = {scale_pos_weight:.2f}")

xgb = XGBClassifier(
    n_estimators      = 200,
    max_depth         = 6,
    learning_rate     = 0.05,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    scale_pos_weight  = scale_pos_weight,
    eval_metric       = "logloss",
    random_state      = RANDOM_STATE,
    n_jobs            = -1
)

xgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50
)

xgb_proba   = xgb.predict_proba(X_val)[:, 1]
xgb_logloss = log_loss(y_val, xgb_proba)
xgb_auc     = roc_auc_score(y_val, xgb_proba)

print("")
print("XGBoost")
print(f"  Log Loss : {xgb_logloss:.4f}")
print(f"  ROC-AUC  : {xgb_auc:.4f}")

## Step 5 — LightGBM

LightGBM is another gradient boosting framework, generally **faster** than XGBoost
because it grows trees leaf-wise instead of level-wise.

For large datasets like KKBOX (970k rows), LightGBM is often preferred.

 automatically adjusts weights inversely proportional to class frequencies,
which is equivalent to  in XGBoost.

In [ ]:
lgbm = LGBMClassifier(
    n_estimators     = 200,
    max_depth        = 6,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    class_weight     = "balanced",
    random_state     = RANDOM_STATE,
    n_jobs           = -1,
    verbose          = -1
)

lgbm.fit(X_train, y_train)

lgbm_proba   = lgbm.predict_proba(X_val)[:, 1]
lgbm_logloss = log_loss(y_val, lgbm_proba)
lgbm_auc     = roc_auc_score(y_val, lgbm_proba)

print("LightGBM")
print(f"  Log Loss : {lgbm_logloss:.4f}")
print(f"  ROC-AUC  : {lgbm_auc:.4f}")

# --- Compare all three models ---
results = pd.DataFrame({
    "Model":    ["Logistic Regression", "XGBoost", "LightGBM"],
    "Log Loss": [lr_logloss, xgb_logloss, lgbm_logloss],
    "ROC-AUC":  [lr_auc,     xgb_auc,     lgbm_auc]
})
results = results.sort_values("Log Loss")
print("")
print("=== Model Comparison ===")
print(results.to_string(index=False))

## Step 6 — Cross-Validation

A single train/val split can be lucky or unlucky depending on which rows end up in each set.
**5-fold Stratified Cross-Validation** gives a more reliable estimate:
- Split data into 5 equal folds
- Train on 4 folds, validate on 1 fold
- Repeat 5 times, each fold serves as validation once
- Report mean ± std of log_loss

We run CV on XGBoost since it performed best.

In [ ]:
# StratifiedKFold preserves the churn ratio in every fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# cross_val_score returns neg_log_loss (negative because sklearn maximizes scores)
cv_scores = cross_val_score(
    XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1
    ),
    X, y,
    cv=skf,
    scoring="neg_log_loss",  # sklearn convention: negate so higher = better
    n_jobs=-1
)

cv_logloss = -cv_scores  # convert back to positive log_loss
print(f"CV Log Loss per fold : {cv_logloss.round(4)}")
print(f"Mean  : {cv_logloss.mean():.4f}")
print(f"Std   : {cv_logloss.std():.4f}")
print(f"95% CI: [{cv_logloss.mean() - 2*cv_logloss.std():.4f}, {cv_logloss.mean() + 2*cv_logloss.std():.4f}]")

## Step 7 — Feature Importance

XGBoost computes **feature importance** as the total gain (reduction in loss)
attributed to each feature across all trees.

Features with high importance are the most useful for predicting churn.
This helps us understand the model and potentially remove low-importance features.

In [ ]:
importances = pd.Series(xgb.feature_importances_, index=X_train.columns)
top15 = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(9, 6))
top15.sort_values().plot(kind="barh", color="steelblue")
plt.xlabel("Feature Importance (Gain)")
plt.title("Top 15 Most Important Features — XGBoost")
plt.tight_layout()
plt.show()

print("Top 15 features:")
print(top15.to_string())

## Step 8 — Confusion Matrix & Classification Report

The **confusion matrix** shows how many predictions were correct vs wrong:

|  | Predicted: No Churn | Predicted: Churn |
|---|---|---|
| **Actual: No Churn** | True Negative (TN) | False Positive (FP) |
| **Actual: Churn** | False Negative (FN) | True Positive (TP) |

For churn prediction, **False Negatives are costly** — missing a churner means losing a customer.

**Precision** = TP / (TP + FP) — of all predicted churners, how many actually churned?
**Recall** = TP / (TP + FN) — of all actual churners, how many did we catch?

With imbalanced data, accuracy is misleading. Focus on precision/recall for the churn class.

In [ ]:
# Predict class labels at threshold 0.5
xgb_pred = (xgb_proba >= 0.5).astype(int)

# --- Confusion Matrix ---
cm = confusion_matrix(y_val, xgb_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred: No Churn", "Pred: Churn"],
            yticklabels=["Actual: No Churn", "Actual: Churn"])
plt.title("Confusion Matrix — XGBoost (threshold=0.5)")
plt.tight_layout()
plt.show()

# --- Classification Report ---
print(classification_report(y_val, xgb_pred, target_names=["No Churn", "Churn"]))

## Step 9 — ROC Curve

The **ROC curve** plots True Positive Rate vs False Positive Rate at every possible threshold.

**AUC (Area Under Curve)**:
- 0.5 = random guessing (diagonal line)
- 1.0 = perfect model
- > 0.85 is generally considered good for churn prediction

Plotting all three models together lets us compare their discriminative power across all thresholds.

In [ ]:
from sklearn.metrics import roc_curve

plt.figure(figsize=(8, 6))

for name, proba in [("Logistic Regression", lr_proba),
                    ("XGBoost",             xgb_proba),
                    ("LightGBM",            lgbm_proba)]:
    fpr, tpr, _ = roc_curve(y_val, proba)
    auc = roc_auc_score(y_val, proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.4f})")

# Diagonal = random classifier baseline
plt.plot([0, 1], [0, 1], "k--", label="Random (AUC = 0.5000)")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — All Models")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## Step 10 — Summary

### Model Performance Comparison

| Model | Log Loss | ROC-AUC | Notes |
|---|---|---|---|
| Logistic Regression | (see output) | (see output) | Baseline, linear only |
| XGBoost | (see output) | (see output) | Handles non-linearity, class imbalance |
| LightGBM | (see output) | (see output) | Faster, similar accuracy |

### Key Findings
-  and  are the strongest churn signals
-  and  capture temporal disengagement
-  and  are useful engineered ratio features
- The dataset is highly imbalanced (~9% churn) —  is essential

### Next Steps
- Add **user_logs features** (listening behavior: total_secs, num_100, completion_rate)
- **Hyperparameter tuning** with Optuna or GridSearchCV
- Try **threshold tuning** to optimize precision/recall tradeoff
- Consider **ensemble** of XGBoost + LightGBM predictions

## Step 11 — Generate Submission File

The competition requires predicting churn probability for users in `sample_submission_v2.csv`.

We use the **best model (XGBoost)** to generate predictions, then save to a CSV file in the format required by Kaggle:
```
msno,is_churn
user_id_1,0.123
user_id_2,0.456
...
```

> **Note:** The test set users may not all appear in `members` or `transactions`. We apply the same preprocessing pipeline and fill missing values with 0.

In [ ]:
# --- Load test set ---
test = pd.read_csv("../data/sample_submission_v2.csv")
print(f"Test set shape: {test.shape}")
display(test.head(3))

# --- Apply same preprocessing pipeline to test users ---
df_test = test[["msno"]].merge(members,   on="msno", how="left")
df_test = df_test.merge(trans_agg,         on="msno", how="left")
df_test = df_test.merge(adv_agg,           on="msno", how="left", suffixes=("","_adv"))
df_test = df_test.merge(last_txn,          on="msno", how="left")
df_test = df_test.merge(temporal_feat,     on="msno", how="left")
if logs_feat is not None:
    df_test = df_test.merge(logs_feat, on="msno", how="left")

# Encode same categorical columns
df_test = pd.get_dummies(df_test, columns=["city", "gender", "registered_via"], dummy_na=True)
df_test = df_test.fillna(0)

# --- Align columns: test must have same features as train ---
# Add any missing columns (filled with 0), drop any extra columns
train_cols = X_train.columns.tolist()
for col in train_cols:
    if col not in df_test.columns:
        df_test[col] = 0
X_test = df_test[train_cols]  # keep only train feature columns, in same order

print(f"Test feature matrix: {X_test.shape}")

# --- Predict churn probability ---
test_proba = xgb.predict_proba(X_test)[:, 1]

# --- Build submission DataFrame ---
submission = pd.DataFrame({
    "msno":     test["msno"],
    "is_churn": test_proba
})

print(f"\nSubmission shape: {submission.shape}")
print(f"Predicted churn rate: {test_proba.mean():.4f}")
display(submission.head())

# --- Save to CSV ---
submission.to_csv("../data/submission.csv", index=False)
print("\nSaved to ../data/submission.csv")